In [3]:

def is_variable(x):
    return isinstance(x, str) and x[0].islower()

def apply_substitution(term, substitution):
    if isinstance(term, str):
        while term in substitution:
            term = substitution[term]
        return term

    return (term[0], tuple(
        apply_substitution(arg, substitution)
        for arg in term[1]
    ))

def occurs_check(var, term, substitution):
    term = apply_substitution(term, substitution)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(
            occurs_check(var, arg, substitution)
            for arg in term[1]
        )

    return False

def unify(x, y, substitution=None):
    if substitution is None:
        substitution = {}

    x = apply_substitution(x, substitution)
    y = apply_substitution(y, substitution)

    if x == y:
        return substitution

    if is_variable(x):
        if occurs_check(x, y, substitution):
            return None
        substitution[x] = y
        return substitution

    if is_variable(y):
        if occurs_check(y, x, substitution):
            return None
        substitution[y] = x
        return substitution

    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return None

        for a, b in zip(x[1], y[1]):
            substitution = unify(a, b, substitution)

            if substitution is None:
                return None

        return substitution

    return None


# Example expressions:
# Knows(John, x) and Knows(y, Mary)

expr1 = ("Knows", ("John", "x"))
expr2 = ("Knows", ("y", "Mary"))

result = unify(expr1, expr2)

if result is not None:
    print("Unification Successful")
    print("Most General Unifier (MGU):")

    for variable, value in result.items():
        print(variable, "=", value)
else:
    print("Unification Failed")


Unification Successful
Most General Unifier (MGU):
y = John
x = Mary
